## 환경 세팅

In [ ]:
%pip install -q -U transformers datasets accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.4/491.4 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 81.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 82.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 100.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does n

In [ ]:
%pip install -q -U trl peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.0/348.0 kB 19.2 MB/s eta 0:00:00


## 데이터셋 불러오기




In [ ]:
from datasets import load_dataset

raw_datasets = load_dataset("sionic-ai/ko-dpo-mix-7k-trl-style")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/425 [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/6389 [00:00<?, ? examples/s]

In [ ]:
from datasets import DatasetDict

# remove this when done debugging
indices = range(0,500)
test_indices = range(500,550)

dataset_dict = {"train": raw_datasets["train"].select(indices),
                "test": raw_datasets["train"].select(test_indices)}

raw_datasets = DatasetDict(dataset_dict)
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['prompt', 'chosen', 'rejected', 'history'],
        num_rows: 500
    })
    test: Dataset({
        features: ['prompt', 'chosen', 'rejected', 'history'],
        num_rows: 50
    })
})

In [ ]:
example = raw_datasets["train"][0]
print(example.keys())

dict_keys(['prompt', 'chosen', 'rejected', 'history'])


## 토크나이저 불러오기


In [ ]:
from transformers import AutoTokenizer

model_id = "Qwen/Qwen3-0.6B-Base"

tokenizer = AutoTokenizer.from_pretrained(model_id)

if tokenizer.pad_token_id is None:
  tokenizer.pad_token_id = tokenizer.eos_token_id

if tokenizer.model_max_length > 100_000:
  tokenizer.model_max_length = 2048

DEFAULT_CHAT_TEMPLATE = "{% for message in messages %}\n{% if message['role'] == 'user' %}\n{{ '<|user|>\n' + message['content'] + eos_token }}\n{% elif message['role'] == 'system' %}\n{{ '<|system|>\n' + message['content'] + eos_token }}\n{% elif message['role'] == 'assistant' %}\n{{ '<|assistant|>\n'  + message['content'] + eos_token }}\n{% endif %}\n{% if loop.last and add_generation_prompt %}\n{{ '<|assistant|>' }}\n{% endif %}\n{% endfor %}"
tokenizer.chat_template = DEFAULT_CHAT_TEMPLATE

tokenizer_config.json:   0%|          | 0.00/9.68k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

## 챗 템플릿 적용하기

In [ ]:
import re
import random
from multiprocessing import cpu_count

def chatml_format(example):
    # Format system
    system = "<|system|>\n" + example.get('system', '') + "<|endoftext|>\n"

    # Format instruction
    prompt = "<|user|>\n" + example['prompt'] + "<|endoftext|>\n<|assistant|>\n"

    # Format chosen answer
    chosen = example['chosen'] + "<|endoftext|>\n"

    # Format rejected answer
    rejected = example['rejected'] + "<|endoftext|>\n"

    return {
        "prompt": system + prompt,
        "chosen": chosen,
        "rejected": rejected,
    }

column_names = list(raw_datasets["train"].features)
raw_datasets = raw_datasets.map(chatml_format,
                                num_proc=cpu_count(),
                                remove_columns=column_names,
                                desc="Applying chat template",)

# create the splits
train_dataset = raw_datasets["train"]
eval_dataset = raw_datasets["test"]

for index in random.sample(range(len(raw_datasets["train"])), 3):
  print(f"Sample {index} of the processed prompt set:\n\n{raw_datasets['train'][index]['prompt']}")
  print(f"Sample {index} of the processed chosen set:\n\n{raw_datasets['train'][index]['chosen']}")
  print(f"Sample {index} of the processed rejected set:\n\n{raw_datasets['train'][index]['rejected']}")

Applying chat template (num_proc=8):   0%|          | 0/500 [00:00<?, ? examples/s]

Applying chat template (num_proc=8):   0%|          | 0/50 [00:00<?, ? examples/s]

Sample 20 of the processed prompt set:

<|system|>
<|endoftext|>
<|user|>
<|system|>
<|endoftext|>
<|user|>
"어제"가 60년대 반문화의 낙관주의와 반란에서 어떻게 벗어나는지 구체적인 예를 들어 설명해 주실 수 있나요?<|endoftext|>
<|assistant|>
<|endoftext|>
<|assistant|>

Sample 20 of the processed chosen set:

"어제"가 60년대 반문화의 낙관주의와 반란에서 어떻게 벗어나는지를 이해하기 위해서는, 당시의 더 널리 퍼진 경향과 비교하여 가사에서 표현된 태도와 주제를 살펴보는 것이 도움이 된다.

1. 과거의 이상화 대 진보와 더 나은 미래에 대한 탐구: 1960년대 반문화의 중요한 특징은 해방, 평등, 자결을 요구하는 음악, 예술, 문학으로 대표되는 사회 변화와 더 나은 미래를 추구하는 것이었다. 그러나 "어제"는 더 단순한 과거에 대한 향수를 통해 정반대의 주장을 한다:

   - "어제, 모든 나의 문제는 너무 멀리 있는 것 같았어/ 이제는 그것들이 여기 머물 것 같은데/ 오, 나는 어제를 믿어"

   이 구절은 시간을 되돌리고 더 행복하고 문제 없는 과거로 돌아가고 싶다는 아픈 욕망을 전달하며, 반문화 운동의 낙관적이고 미래 지향적인 초점과는 정반대이다.

2. 후회와 취약성 대 반란과 자기 확신: 1960년대 반문화는 자기 확신과 개성을 촉진하는 강력한 음악과 메시지를 보았다. 반면, "어제"는 혼란과 후회의 감각을 묘사한다:

   - "그녀가 왜 떠나야 했는지, 나는 모르겠어, 그녀는 말하지 않았어/ 나는 뭔가 잘못 말했어, 이제 나는 어제를 그리워해"

   가수의 실패한 관계에 대한 실망과 궁극적인 과거에 대한 그리움은 젊은 가수가 앞으로 나아가고, 사회적 규범에 도전하며, 동조를 거부하도록 격려했을 다른 운동들과는 다르다.

"어제"가 반문화의 몇 가지 공통 주제에서 벗어나지만, 그것

In [ ]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['prompt', 'chosen', 'rejected'],
        num_rows: 500
    })
    test: Dataset({
        features: ['prompt', 'chosen', 'rejected'],
        num_rows: 50
    })
})

## 모델 학습 준비하기

In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import AutoTokenizer, AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto",
    device_map="auto",
)

lora_config = LoraConfig(
    task_type="CAUSAL_LM",
    r=64,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

trainable params: 18,350,080 || all params: 614,400,000 || trainable%: 2.9867


In [ ]:
from trl import DPOTrainer, DPOConfig

output_dir = 'data/test_model'

training_args = DPOConfig(
    fp16=True,
    do_eval=True,
    eval_strategy="epoch",
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    learning_rate=3.0e-04,
    log_level="info",
    logging_steps=5,
    logging_strategy="steps",
    lr_scheduler_type="cosine",
    max_steps=-1,
    num_train_epochs=2,
    output_dir=output_dir,
    overwrite_output_dir=True,
    per_device_eval_batch_size=1,
    per_device_train_batch_size=2,
    save_strategy="epoch",
    save_total_limit=1,
    seed=42,
    report_to="tensorboard"
)

trainer = DPOTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        processing_class=tokenizer
    )

Extracting prompt in train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (2193 > 2048). Running this sequence through the model will result in indexing errors


Extracting prompt in eval dataset:   0%|          | 0/50 [00:00<?, ? examples/s]

Applying chat template to eval dataset:   0%|          | 0/50 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/50 [00:00<?, ? examples/s]

Using auto half precision backend
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


## 학습해보기!

In [ ]:
train_result = trainer.train()

***** Running training *****
  Num examples = 500
  Num Epochs = 5
  Instantaneous batch size per device = 2
  Total train batch size (w. parallel, distributed & accumulation) = 16
  Gradient Accumulation steps = 8
  Total optimization steps = 155
  Number of trainable parameters = 18,350,080
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Epoch,Training Loss,Validation Loss,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected,Logits/chosen,Logits/rejected
1,0.711900,0.656276,0.115169,-0.266527,0.620000,0.381696,-810.047058,-795.304504,-2.923690,-2.928061
2,0.132100,0.838596,-1.962867,-2.786447,0.560000,0.823580,-830.827515,-820.503601,-3.178094,-3.186997
3,0.043400,0.912146,-2.018129,-3.021090,0.580000,1.002961,-831.380066,-822.850098,-3.169283,-3.172080
4,0.020700,0.918933,-1.795698,-2.883906,0.580000,1.088208,-829.155762,-821.478210,-3.208541,-3.208163



***** Running Evaluation *****
  Num examples = 50
  Batch size = 1
Saving model checkpoint to data/test_model/checkpoint-32
loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen3-0.6B-Base/snapshots/11214f7f3465775dcce23c3752ecea5a42ee0ddc/config.json
Model config Qwen3Config {
  "architectures": [
    "Qwen3ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "eos_token_id": 151643,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 1024,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "max_position_embeddings": 32768,
  "max_window_layers": 28,
  "model_type": "qwen3",
  "num_attention_heads": 16,
  "num_hidden_layers": 28,
  "num_key_value_heads": 8,
  "rms_norm_eps": 1e-06,
  "rope_scaling": null,
  "rope_theta": 1000000,
  "sliding_window": null,
  "tie_word_embeddings": true,
  "torch_dtype": "bfloat16",
  "transformers_version": "4.51.3",
  "use_cache": true,
  "

In [ ]:
trainer.model.save_pretrained(output_dir)
trainer.processing_class.save_pretrained(output_dir)

loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen3-0.6B-Base/snapshots/11214f7f3465775dcce23c3752ecea5a42ee0ddc/config.json
Model config Qwen3Config {
  "architectures": [
    "Qwen3ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "eos_token_id": 151643,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 1024,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "max_position_embeddings": 32768,
  "max_window_layers": 28,
  "model_type": "qwen3",
  "num_attention_heads": 16,
  "num_hidden_layers": 28,
  "num_key_value_heads": 8,
  "rms_norm_eps": 1e-06,
  "rope_scaling": null,
  "rope_theta": 1000000,
  "sliding_window": null,
  "tie_word_embeddings": true,
  "torch_dtype": "bfloat16",
  "transformers_version": "4.51.3",
  "use_cache": true,
  "use_sliding_window": false,
  "vocab_size": 151936
}

Trainer.tokenizer is now deprecated. You should use Trainer.processing_c

('data/test_model/tokenizer_config.json',
 'data/test_model/special_tokens_map.json',
 'data/test_model/vocab.json',
 'data/test_model/merges.txt',
 'data/test_model/added_tokens.json',
 'data/test_model/tokenizer.json')

## 학습한 모델로 생성해보기

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

output_dir = 'data/test_model'
tokenizer = AutoTokenizer.from_pretrained(output_dir)
model = AutoModelForCausalLM.from_pretrained(output_dir, device_map="auto")

loading file vocab.json
loading file merges.txt
loading file tokenizer.json
loading file added_tokens.json
loading file special_tokens_map.json
loading file tokenizer_config.json
loading file chat_template.jinja
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen3-0.6B-Base/snapshots/11214f7f3465775dcce23c3752ecea5a42ee0ddc/config.json
Model config Qwen3Config {
  "architectures": [
    "Qwen3ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "eos_token_id": 151643,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 1024,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "max_position_embeddings": 32768,
  "max_window_layers": 28,
  "model_type": "qwen3",
  "num_attention_heads": 16,
  "num_hidden_layers": 28,
  "num_key_value_heads": 8,
  "rms_norm_eps"

In [ ]:
tokenizer.chat_template = DEFAULT_CHAT_TEMPLATE

In [ ]:
import torch

# We use the tokenizer's chat template to format each message - see https://huggingface.co/docs/transformers/main/en/chat_templating
messages = [
    {"role": "system", "content": ""},
    {"role": "user", "content": "정보 이론이란 무엇인가요?"},
]

# prepare the messages for the model
input_ids = tokenizer.apply_chat_template(messages, truncation=True, add_generation_prompt=True, return_tensors="pt").to("cuda")

# inference
outputs = model.generate(
        input_ids=input_ids,
        max_new_tokens=256,
        do_sample=True,
        temperature=1.0,
        top_p=0.95
)
print(tokenizer.batch_decode(outputs, skip_special_tokens=True)[0])

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


<|system|>

<|user|>
정보 이론이란 무엇인가요?
<|assistant|>
정보 이론은 정보를 처리, 분석하고 윤리적이고 안전한 방식으로 사용하는 기술과 방법을 보다 넓게 정의하기 위해 개발된 새로운 지식과 이론을 다루는 분야입니다. 정보 이론의 주요 범위에는 정보의 양, 정보의 질, 정보의 사용 및 정보의 관리를 포함합니다. 이론은 정보 분석, 비즈니스 요약, 인공지능, 데이터 분석, 생물정보학, 물리정보학, 천문학 등의 분야에서 주로 이론적 접근 방식을 보강하며, 이론적 접근 방식은 실제 사례에서의 데이터 분석과 유사하게 적용될 수 있습니다.

정보 이론은 정보를 통합하고 변형하여 정보를 효율적으로 활용하기 위해 주요 이론적 접근 방식을 제공합니다. 이론은 다음과 같은 주요 접근 방식을 포함하여 사용됩니다:

1. 통제적 접근 방식: 이 접근 방식은 주어진 상태에서 가능한 가능한 이입을 평가하고 가능한 가능한 결정을 분석하여 최적의 결과를 생성하는 접근 방식
